# Lab 12: Position Models as Geometric Transformations

**Computer Vision Course**

Building on corner detection and matching (Lab 11), today you'll learn how to estimate **geometric transformations** between images.

**What you'll learn:**
- Homography estimation from matched keypoints
- SIFT feature detection and matching
- RANSAC for robust estimation
- Perspective transformation using cv2.warpPerspective

**What you'll build:**
- Pose correction (dewarp distorted images)
- Augmented reality (replace objects in scenes)
- Panorama stitching (combine overlapping images)

**Why this matters:**
Geometric transformations are everywhere in computer vision:
- Document scanning and correction
- Augmented Reality applications
- Panoramic photography
- 3D reconstruction
- Robot navigation

**Connection to previous labs:**
- Lab 11: Corner detection and patch matching
- Lab 12 (today): Estimate transformations from matches

## Setup

In [ ]:
"""
Computer Vision Course - Lab 12: Geometric Transformations

This cell sets up the environment.
Works automatically for both local and Google Colab!
"""

import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

print("=" * 60)
print("Computer Vision - Lab 12: Geometric Transformations")
print("=" * 60)

if IN_COLAB:
    print("\n🔵 Running on Google Colab")
    print("-" * 60)
    
    if not os.path.exists('computer-vision'):
        print("📥 Cloning repository...")
        !git clone https://github.com/mjck/computer-vision.git
        print("✓ Repository cloned successfully")
    else:
        !git -C computer-vision pull
        print("✓ Repository already exists")
    
    %cd computer-vision/labs/lab12_transformations
    print(f"✓ Current directory: {os.getcwd()}")
    
    sys.path.insert(0, '/content/computer-vision')
    print("✓ Python path configured")
    
    print("-" * 60)
    print("🟢 Colab setup complete!\n")
    
else:
    print("\n🟢 Running locally")
    print("-" * 60)
    print(f"✓ Current directory: {os.getcwd()}")
    
    repo_root = os.path.abspath('../..')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f"✓ Repository root: {repo_root}")
    
    print("-" * 60)
    print("🟢 Local setup complete!\n")

print("=" * 60)
print("✅ Environment ready!")
print("=" * 60)

## Import Libraries

In [ ]:
import numpy as np
import cv2

# Import course utilities
try:
    from sdx import cv_imread, cv_imshow, cv_grayread
    print("✓ sdx module loaded")
except ImportError as e:
    print(f"❌ Could not import sdx: {e}")
    raise

print("✓ All imports successful")

---
## Choosing an Image Pair

We have 4 different scenarios to explore:

1. **`detection`** — Simple example to understand the workflow
2. **`correction`** — For Activity 1 (pose correction)
3. **`augmented`** — For Activity 2 (augmented reality)
4. **`panorama`** — For Activity 3 (panorama stitching)

Start with `detection` to learn the pipeline, then switch for activities!

In [ ]:
# Choose your scenario
SOURCE = 'detection'  # Options: 'detection', 'correction', 'augmented', 'panorama'

print(f"✓ Selected scenario: {SOURCE}")
print("\nNote: You'll change this for each activity later!")

---
## Loading Images

### Terminology

We use OpenCV's keypoint framework terminology:

- **Train image:** The "object" we're looking for (reference)
- **Query image:** The "scene" where we search for the object

This naming matches OpenCV's SIFT/matching documentation.

### Load Train Image (Object)

In [ ]:
train = cv_grayread(f'{SOURCE}-train.png')

train_height, train_width = train.shape

print(f"Train image: {train.shape}")
cv_imshow(train)
print(f"This is the 'object' we'll search for")

### Load Query Image (Scene)

In [ ]:
query = cv_grayread(f'{SOURCE}-query.png')

query_height, query_width = query.shape

print(f"Query image: {query.shape}")
cv_imshow(query)
print(f"This is the 'scene' where we'll find the object")

---
## Extracting Keypoints and Descriptors

**SIFT (Scale-Invariant Feature Transform):**
- Detects keypoints at multiple scales (Lab 9 concepts!)
- Computes rotation-invariant descriptors
- Each descriptor is a 128-dimensional vector
- Much better than simple patch matching (Lab 11)

**What SIFT does:**
1. Find keypoints (corners) across scale-space
2. Assign dominant orientation to each keypoint
3. Compute descriptor in normalized patch

**Why SIFT is powerful:**
- ✅ Scale-invariant (works at different sizes)
- ✅ Rotation-invariant (works at different angles)
- ✅ Illumination-robust (handles lighting changes)
- ✅ Distinctive (128-dim vectors are unique)

In [ ]:
# Create SIFT detector
sift = cv2.SIFT_create()

# Detect keypoints and compute descriptors
# None = process entire image (no mask)
train_keypoints, train_descriptors = sift.detectAndCompute(train, None)
query_keypoints, query_descriptors = sift.detectAndCompute(query, None)

print(f"Train: {len(train_keypoints)} keypoints detected")
print(f"Query: {len(query_keypoints)} keypoints detected")
print(f"\nDescriptor dimensionality: {train_descriptors.shape[1]}")
print("(Each keypoint described by a 128-dimensional vector)")

---
## Keypoint Matching Based on Descriptor Similarity

Instead of comparing patches (Lab 11), we compare **SIFT descriptors**.

**FLANN (Fast Library for Approximate Nearest Neighbors):**
- Efficient algorithm for finding similar descriptors
- Much faster than brute-force search
- Used for large-scale matching

**k-NN Matching (k=2):**
- For each query descriptor, find 2 nearest train descriptors
- Why 2? To apply Lowe's ratio test (next step)

In [ ]:
# Create FLANN matcher
matcher = cv2.FlannBasedMatcher()

# For each query descriptor, find k=2 nearest matches
matches = matcher.knnMatch(query_descriptors, train_descriptors, k=2)

print(f"Found {len(matches)} query keypoints")
print("Each has 2 candidate matches from train image")

### Lowe's Ratio Test

**The problem:** Many matches are ambiguous or wrong!

**Lowe's ratio test (from SIFT paper):**
- Look at the two best matches for each query descriptor
- If **best_distance < 0.8 × second_best_distance**, accept the match
- Otherwise, reject both (ambiguous match)

**Why it works:**
- Good matches: nearest neighbor much closer than second nearest
- Bad matches: both neighbors similarly distant (ambiguous)
- Threshold 0.8 is empirically optimal (from SIFT paper)

In [ ]:
# Binary mask for visualization
# (which matches to draw)
matches_mask = []

# List of good matches (passed ratio test)
good_matches = []

# Each element is (first_match, second_match)
for first, second in matches:
    # Ratio test: is first significantly better than second?
    if first.distance < 0.8 * second.distance:
        # Good match! Keep it
        matches_mask.append((1, 0))  # Draw first, not second
        good_matches.append(first)
    else:
        # Ambiguous match, reject both
        matches_mask.append((0, 0))  # Draw neither

print(f"Total matches: {len(matches)}")
print(f"Good matches (passed ratio test): {len(good_matches)}")
print(f"Rejected (ambiguous): {len(matches) - len(good_matches)}")
print(f"\nAcceptance rate: {100 * len(good_matches) / len(matches):.1f}%")

### Visualize Matches

In [ ]:
# Draw matches (only good ones, due to matches_mask)
matches_output = cv2.drawMatchesKnn(
    query, query_keypoints,      # Query image and keypoints
    train, train_keypoints,      # Train image and keypoints  
    matches,                     # All matches
    None,                        # Output image (None = create new)
    matchesMask=matches_mask     # Which matches to draw
)

cv_imshow(matches_output)
print("Green lines connect matched keypoints")
print("Only matches that passed Lowe's ratio test are shown")

---
## Estimating Pose (Homography)

**The goal:** Find the geometric transformation between train and query images.

**Homography (H):**
- A 3×3 matrix that maps points from one image to another
- Represents perspective transformation
- Can handle rotation, scale, translation, and perspective distortion

**What homography does:**
```
[x']     [h11 h12 h13]   [x]
[y']  =  [h21 h22 h23] × [y]
[1 ]     [h31 h32 h33]   [1]
```

Then: x_final = x'/1, y_final = y'/1 (homogeneous coordinates)

**Use case:** Maps train image coordinates → query image coordinates

In [ ]:
# Prepare point correspondences from good matches
pose_src = []  # Train image points
pose_dst = []  # Query image points

for match in good_matches:
    # trainIdx: index in train_keypoints
    # queryIdx: index in query_keypoints
    pose_src.append(train_keypoints[match.trainIdx].pt)
    pose_dst.append(query_keypoints[match.queryIdx].pt)

# Convert to NumPy arrays (required by findHomography)
pose_src = np.array(pose_src)
pose_dst = np.array(pose_dst)

print(f"Using {len(pose_src)} point correspondences")
print(f"Source points shape: {pose_src.shape}")
print(f"Destination points shape: {pose_dst.shape}")

### RANSAC for Robust Estimation

**The problem:** Some matches are still wrong (outliers)!

**RANSAC (Random Sample Consensus):**
1. Randomly sample small subset of matches
2. Fit homography to this subset
3. Count how many other matches agree (inliers)
4. Repeat many times
5. Keep homography with most inliers

**Why it works:**
- Outliers are random → unlikely to be in same random sample
- Correct matches (inliers) → consistent with same homography
- Robust to even 50% outliers!

**cv2.findHomography with RANSAC:**
- Automatically applies RANSAC
- Returns H (homography) and mask (inlier/outlier labels)

In [ ]:
# Estimate homography using RANSAC
H, mask = cv2.findHomography(
    pose_src,      # Source points (train image)
    pose_dst,      # Destination points (query image)
    cv2.RANSAC     # Use RANSAC for robustness
)

# Count inliers
inliers = np.sum(mask)
outliers = len(mask) - inliers

print(f"Homography estimated:")
print(H)
print(f"\nRANSAC results:")
print(f"  Inliers: {inliers} ({100*inliers/len(mask):.1f}%)")
print(f"  Outliers: {outliers} ({100*outliers/len(mask):.1f}%)")
print("\n✓ Homography is robust even with outliers!")

---
## Validating the Homography

To visualize the transformation, we'll:
1. Define a rectangle around the train image
2. Transform it using the homography
3. Draw the transformed rectangle on the query image

This shows where the object appears in the scene!

In [ ]:
# Define rectangle around train image
# Note: OpenCV uses (x, y) ordering!
rectangle_src = [
    (0, 0),                          # Top-left
    (0, train_height),               # Bottom-left
    (train_width, train_height),     # Bottom-right
    (train_width, 0),                # Top-right
]

# Convert to NumPy array with required shape
# perspectiveTransform needs shape: (N, 1, 2) with float32
rectangle_src = np.array([rectangle_src], dtype=np.float32)

print(f"Rectangle corners in train image:")
print(rectangle_src.reshape(4, 2))

### Transform Rectangle

In [ ]:
# Apply homography to rectangle
rectangle_dst = cv2.perspectiveTransform(rectangle_src, H)

print(f"\nRectangle corners in query image (after transformation):")
print(rectangle_dst.reshape(4, 2))
print("\nThese are where the train image corners appear in the query!")

### Draw Transformed Rectangle

In [ ]:
# Convert to BGR for colored drawing
rectangle_output = cv2.cvtColor(query, cv2.COLOR_GRAY2BGR)

# Draw lines between consecutive corners
color = (0, 255, 0)  # Green
thickness = 2

for i in range(4):
    # Draw line from corner i to corner (i+1) mod 4
    pt1 = tuple(rectangle_dst[0][i].astype(int))
    pt2 = tuple(rectangle_dst[0][(i + 1) % 4].astype(int))
    cv2.line(rectangle_output, pt1, pt2, color, thickness)

cv_imshow(rectangle_output)
print("\n✓ Green rectangle shows where train image appears in query!")
print("✓ Notice: can handle rotation, scaling, perspective!")

---
## Loading Color Versions

For the activities, we'll work with color images for better visual results.

In [ ]:
# Load color versions
color_train = cv_imread(f'{SOURCE}-train.png')
color_query = cv_imread(f'{SOURCE}-query.png')

print("Color train image:")
cv_imshow(color_train)

print("\nColor query image:")
cv_imshow(color_query)

print("\n✓ Color images loaded for activities")

---
## Activity 1: Pose Correction

**Goal:** Remove perspective distortion from a document or sign.

**The task:**
1. Switch `SOURCE` to `'correction'`
2. Run the entire notebook up to this point
3. Write code to **correct the perspective distortion**

**Hint:** You want to transform the **query image** to look like the **train image**.
- Train image: undistorted (frontal view)
- Query image: distorted (perspective view)
- Need: inverse transformation!

**Expected output:** [Reference image](https://drive.google.com/file/d/1TfuBFtbgcpvvvxWUWQzDMaCOqG3RCcPN/view?usp=drive_link)

**Think about:**
- What transformation do you need?
- What size should the output be?
- Which cv2 function applies transformations?

In [ ]:
# ── Activity 1: Your code here ──────────────────────────────────────────────

output1 = color_train.copy()  # Replace this line with your code

# ────────────────────────────────────────────────────────────────────────────

cv_imshow(output1)
print("Activity 1: Pose correction")

---
## Activity 2: Augmented Reality

**Goal:** Replace the object in the scene with a different image.

**Setup:** First, load the replacement image.

In [ ]:
# Load the image we'll insert
color_other = cv_imread('images/augmented-other.png')

other_height, other_width, _ = color_other.shape

print(f"Replacement image: {color_other.shape}")
cv_imshow(color_other)
print("This will replace the object in the scene")

**The task:**
1. Switch `SOURCE` to `'augmented'`
2. Run the entire notebook up to this point
3. Write code to **replace the object in the query image with the other image**

**Expected output:** [Reference image](https://drive.google.com/file/d/1l8OMcA73azrcMxQS7lzaSLlOOCY3ouEQ/view?usp=drive_link)

**Think about:**
- What transformation maps `other` → `query`?
- How to avoid overwriting the background?
- Hint: cv2.warpPerspective can take an output image parameter!

In [ ]:
# ── Activity 2: Your code here ──────────────────────────────────────────────

output2 = color_query.copy()

# Write your code here

# ────────────────────────────────────────────────────────────────────────────

cv_imshow(output2)
print("Activity 2: Augmented reality")

---
## Activity 3: Panorama Stitching

**Goal:** Combine two overlapping images into a panorama.

**The task:**
1. Switch `SOURCE` to `'panorama'`
2. Run the entire notebook up to this point
3. Write code to **stitch the train and query images together**

**Expected output:** [Reference image](https://drive.google.com/file/d/1di7bVdMf67LthIU4fW6-6r4_S88Z5B-x/view?usp=drive_link)

**Think about:**
- Query image is already placed in the output
- Where should train image go?
- What transformation aligns train to query?
- Need: inverse homography!

In [ ]:
# ── Activity 3: Your code here ──────────────────────────────────────────────

# Create large output canvas
output3 = np.zeros((round(1.3 * query_height), round(1.4 * query_width), 3), dtype=np.uint8)

# Place query image in bottom-right
output3[-query_height:, -query_width:, :] = color_query

# Write your code here to add the train image

# ────────────────────────────────────────────────────────────────────────────

cv_imshow(output3)
print("Activity 3: Panorama stitching")

---
## Summary

### What You Learned

1. **SIFT Features**
   - Scale and rotation invariant
   - 128-dimensional descriptors
   - Much better than simple patches

2. **Feature Matching**
   - FLANN for efficient nearest neighbor search
   - Lowe's ratio test to filter ambiguous matches
   - Typically accept ~30-50% of initial matches

3. **Homography Estimation**
   - 3×3 matrix for perspective transformation
   - Estimated from point correspondences
   - RANSAC for robust estimation (handles outliers)

4. **Transformation Applications**
   - Pose correction: remove perspective distortion
   - Augmented reality: replace objects in scenes
   - Panorama stitching: combine overlapping images

### Key Functions

**SIFT:**
```python
sift = cv2.SIFT_create()
keypoints, descriptors = sift.detectAndCompute(image, None)
```

**Matching:**
```python
matcher = cv2.FlannBasedMatcher()
matches = matcher.knnMatch(desc1, desc2, k=2)
```

**Homography:**
```python
H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC)
```

**Warping:**
```python
output = cv2.warpPerspective(image, H, (width, height))
```

### Real-World Applications

**Document Scanning:**
- Detect document corners
- Estimate homography
- Dewarp to frontal view

**Augmented Reality:**
- Detect marker/object
- Estimate pose
- Overlay virtual content

**Panoramas:**
- Match overlapping images
- Estimate transformations
- Warp and blend

**3D Reconstruction:**
- Multiple views of scene
- Estimate relative poses
- Triangulate 3D points

### Connection to Previous Labs

**Lab 11 (Corners):**
- Simple patch matching
- Limited to translation
- Not rotation/scale invariant

**Lab 12 (today):**
- SIFT features (rotation/scale invariant)
- Full perspective transformations
- RANSAC for robustness

**Looking ahead:**
- Structure from Motion (3D reconstruction)
- Visual SLAM (robot navigation)
- Deep learned features